## Data Processing

In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import numpy as np

In [ ]:
# file directories

file_path = "..\\data\\Global Factor Data.parquet"
FEATURES_RAW =  "...\\data\\features_clean.parquet"
TARGET_FILE = "..\\data\\target.parquet"
FEATURES_FLAG = "..\\data\\features_with_flags.parquet"
FEATURES_FINAL = "..\\data\\features_ranked.parquet"

# Read full parquet file into Arrow Table
table = pq.read_table(file_path)
df = table.to_pandas()

display(df.head())

schema = pq.read_schema(file_path)
metadata = pq.read_metadata(file_path)
n_rows = metadata.num_rows
n_columns = len(schema.names)

print(f"Row count:{n_rows}, Column count: {n_columns}")

In [ ]:
parquet_file = pq.ParquetFile(file_path)
results = []

for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):

    batch_table = pa.Table.from_batches([batch])
    df_batch = batch_table.to_pandas()

    nan_counts = df_batch.isnull().sum(axis=1)
    results.append(nan_counts)

    print(f"Batch {i + 1} processed | rows: {len(df_batch)}")

nan_per_row = pd.concat(results, ignore_index=True)
print(f"Total rows: {len(nan_per_row)}")
print(f"Rows with zero NaNs: {(nan_per_row == 0).sum()}")
print(f"Rows with at least one NaN: {(nan_per_row > 0).sum()}")
print(f"Average NaNs per row: {nan_per_row.mean():.2f}")
print(f"Max NaNs in a single row: {nan_per_row.max()}")

In [ ]:
raw_columns = schema.names
raw_n_rows  = metadata.num_rows

COLUMNS_TO_REMOVE: set[str] = {

    # Identifiers and keys
    "permno", "permco", "eom",

    # data-quality flags and others
    "obs_main", "exch_main", "primary_sec", "common", "size_grp",
    "ret_lag_dif", "adjfct", "bidask", "comp_tpci", "crsp_shrcd",
    "comp_exchg", "crsp_exchcd", "source_crsp", "curcd",

    # Target variable (isolated separately)
    "ret_exc_lead1m",

    # returns (direct look-ahead risk)
    "ret_exc", "ret", "ret_local",

    # Raw unscaled accounting numbers (dimensionless variants are retained)
    "enterprise_value", "book_equity", "assets", "sales", "net_income",

    # Exchange rates and industry codes
    "fx", "gics", "naics", "sic", "ff49",

    # Raw price and volume data
    "prc", "prc_local", "prc_high", "prc_low", "shares", "tvol",
}

TARGET_COLUMN = "ret_exc_lead1m"

# Partition the raw columns into feature and target lists. The feature list
# preserves the original parquet ordering for column-projection efficiency
feature_cols = [c for c in raw_columns if c not in COLUMNS_TO_REMOVE and c != TARGET_COLUMN]
target_cols = [TARGET_COLUMN] if TARGET_COLUMN in raw_columns else []

print(f"Columns removed: {len(COLUMNS_TO_REMOVE & set(raw_columns))}")
print(f"Features kept: {len(feature_cols)}")
print(f"Target present: {bool(target_cols)}")

In [ ]:
# Stream the raw parquet in batches, projecting onto feature and target columns
# and writing to separate output files. Lazy writer creation ensures the output
# schemas are inferred from the actual projected batch.

parquet_file = pq.ParquetFile(file_path)
feature_writer = None
target_writer = None
total_rows = 0

try:
    for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):
        batch_table = pa.Table.from_batches([batch])

        feature_table = batch_table.select(feature_cols)
        if feature_writer is None:
            feature_writer = pq.ParquetWriter(FEATURES_RAW, feature_table.schema, compression="snappy")
        feature_writer.write_table(feature_table)

        if target_cols:
            target_table = batch_table.select(target_cols)
            if target_writer is None:
                target_writer = pq.ParquetWriter(TARGET_FILE, target_table.schema, compression="snappy")
            target_writer.write_table(target_table)

        total_rows += batch_table.num_rows
        print(f"Batch {i + 1:2d}  rows={batch_table.num_rows:>8,}  cumulative={total_rows:>10,}")
finally:
    if feature_writer is not None:
        feature_writer.close()
    if target_writer is not None:
        target_writer.close()